# Assignment 3: Local Association Matrix 

**Student names**: Daniel Høyland <br>
**Group number**: 4 <br>
**Date**: 06.10.2025

## Important notes
Please read and follow these rules. Submissions that do not fulfill them may be returned.
1. You may work in groups of maximum 2 students.
2. Submit in **.ipynb** format only.
3. The assignment must be typed. Handwritten answers are not accepted.

**Due date**: 12.10.2025 23:59

### What you will do 
- Build a **local association matrix** from Cranfield collection.
- Compute the **normalized association matrix**.
- Use the normalized matrix to **identify neighborhood terms** for expansion for given queries.


---
## Dataset

You will use the **Cranfield** dataset, provided in this file:

- `cran.all.1400`: The document collection (1400 documents)

**The code to parse the file is ready — just update the cran file path to match your own file location. Use the docs variable in your code for the parsed file**


### Load and parse documents (provided)

Run the cell to parse the Cranfield documents. Update the path so it points to your `cran.all.1400` file.

In [1]:
# Read 'cran.all.1400' and parse the documents into a suitable data structure

CRAN_PATH = r"cran.all.1400"  # <-- change this!

def parse_cranfield(path):
    docs = {}
    current_id = None
    current_field = None
    buffers = {"T": [], "A": [], "B": [], "W": []}
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.rstrip("\n")
            if line.startswith(".I "):
                if current_id is not None:
                    docs[current_id] = {
                        "id": current_id,
                        "title": " ".join(buffers["T"]).strip(),
                        "abstract": " ".join(buffers["W"]).strip()
                    }
                current_id = int(line.split()[1])
                buffers = {k: [] for k in buffers}
                current_field = None
            elif line.startswith("."):
                tag = line[1:].strip()
                current_field = tag if tag in buffers else None
            else:
                if current_field is not None:
                    buffers[current_field].append(line)
    if current_id is not None:
        docs[current_id] = {
            "id": current_id,
            "title": " ".join(buffers["T"]).strip(),
            "abstract": " ".join(buffers["W"]).strip()
        }
    print(f"Parsed {len(docs)} documents.")
    return docs

docs = parse_cranfield(CRAN_PATH)

Parsed 1400 documents.


## 3.1  Local association matrix

For the given Cranfield document collection in cran.all.1400 construct a local association matrix to identify association clusters. Use the docs variable with the parsed file. Omit stopwords in the STOPWORDS list given below from the vocabulary. 


The correlation factors $c_{u,v}$ between any pair of terms $w_u$ and $w_v$ are defined as  
$c_{u,v} = \sum_{d_j \in D} f_{u,j} \cdot f_{v,j}$  

$f_{u,j}$ is the raw term frequency of $w_u$ in document $d_j$.

### Weighting variants: **scalar** and **metric**

Add two alternative weighting schemes for the matrix (only the formula for assigning the matrix cell value changes):

- **Metric weighting** :
Let $w_u(n,j)$ and $w_v(m,j)$ denote the $n$-th and $m$-th occurrences of terms $w_u$ and $w_v$ in document $d_j$.  
Define a distance function $r(w_u(n,j), w_v(m,j))$ (e.g., $r(i,k) = 1 + |i - k|$).  
Then:

$$
c_{u,v} = \sum_{d_j \in D} \sum_n \sum_m \frac{1}{r(w_u(n,j), w_v(m,j))}
$$


- **Scalar weighting** :
Let $\vec{s}_u = \langle c_{u,x_1}, c_{u,x_2}, \dots, c_{u,x_n} \rangle$ be the neighborhood vector of term $w_u$, and similarly for $w_v$.  
Then:

$$
c_{u,v} = \frac{\vec{s}_u \cdot \vec{s}_v}{|\vec{s}_u| \cdot |\vec{s}_v|}
$$

In [2]:
# TODO: Construct a local association matrix for the cranfield collection. Use both weighting variants.

STOPWORDS = set("""a about above after again against all am an and any are aren't as at be because been
before being below between both but by can't cannot could couldn't did didn't do does doesn't doing don't down
during each few for from further had hadn't has hasn't have haven't having he he'd he'll he's her here here's hers
herself him himself his how how's i i'd i'll i'm i've if in into is isn't it it's its itself let's me more most
mustn't my myself no nor not of off on once only or other ought our ours ourselves out over own same shan't she
she'd she'll she's should shouldn't so some such than that that's the their theirs them themselves then there there's
these they they'd they'll they're they've this those through to too under until up very was wasn't we we'd we'll we're
we've were weren't what what's when when's where where's which while who who's whom why why's with won't would wouldn't
you you'd you'll you're you've your yours yourself yourselves""".split())

# Your code here
import re
import numpy as np
from collections import defaultdict, Counter
from scipy.sparse import csr_matrix

def tokenize(text):
    tokens = re.findall(r"[a-zA-Z]+", text.lower())
    return [t for t in tokens if t not in STOPWORDS]

doc_tokens = {}
vocab = set()

for doc_id, d in docs.items():
    tokens = tokenize(d["title"] + " " + d["abstract"])
    doc_tokens[doc_id] = tokens
    vocab.update(tokens)

vocab = sorted(vocab)
vocab_index = {w: i for i, w in enumerate(vocab)}
V = len(vocab)
print(f"Vocabulary size: {V}")




Vocabulary size: 6934


In [3]:
rows, cols, vals = [], [], []
for row_idx, (doc_id, tokens) in enumerate(doc_tokens.items()):
    counts = Counter(tokens)
    for t, c in counts.items():
        rows.append(row_idx)        
        cols.append(vocab_index[t])
        vals.append(c)

tf_sparse = csr_matrix((vals, (rows, cols)), shape=(len(docs), len(vocab)), dtype=int)


local_assoc_sparse = tf_sparse.T @ tf_sparse
print("Standard local association (sparse) shape:", local_assoc_sparse.shape)


Standard local association (sparse) shape: (6934, 6934)


In [4]:
from sklearn.metrics.pairwise import cosine_similarity

scalar_assoc = cosine_similarity(local_assoc_sparse, dense_output=False)

print("Scalar weighting matrix shape:", scalar_assoc.shape)
print("Example entry:", scalar_assoc[0,1])

Scalar weighting matrix shape: (6934, 6934)
Example entry: 0.03928431631004639


## 3.2 Normalized association matrix

Compute the normalized association matrix from the unnormalized matrix computed above. 

To normalize the matrix use the following formula: <br>
$c'_{u,v} = \frac{c_{u,v}}{c_{u,u} + c_{v,v} - c_{u,v}}$  


In [5]:
#TODO: Compute the normalized association matrix 

# Your code here
def normalize_assoc_matrix(matrix):

    diag = matrix.diagonal()  
    rows, cols = matrix.nonzero()
    data = []

    for i, j in zip(rows, cols):
        denom = diag[i] + diag[j] - matrix[i, j]
        if denom > 0:
            val = matrix[i, j] / denom
        else:
            val = 0.0
        data.append(val)

    norm_matrix = csr_matrix((data, (rows, cols)), shape=matrix.shape)
    return norm_matrix

normalized_assoc_sparse = normalize_assoc_matrix(local_assoc_sparse)

print("Normalized association matrix shape:", normalized_assoc_sparse.shape)
print("Number of nonzeros:", normalized_assoc_sparse.nnz)

Normalized association matrix shape: (6934, 6934)
Number of nonzeros: 3044654


## 3.3 Neighborhood terms

With the help of the normalized local association matrix, identify the neighborhood terms that should be used for expansion for the following queries (queries_assignment3):


In [6]:
# Do not change this code
queries_assignment3 = [
  "gas pressure",
  "structural aeroelastic flight high speed aircraft",
  "heat conduction composite slabs",
  "boundary layer control",
  "compressible flow nozzle",
  "combustion chamber injection",
  "laminar turbulent transition",
  "fatigue crack growth",
  "wing tip vortices",
  "propulsion efficiency"
]

In [7]:
#TODO: Identify neighborhood terms for queries_assignment3

# Your code here

def get_neighborhood_terms_sparse(query, vocab_index, vocab, assoc_matrix, topk=10):
    tokens = tokenize(query)
    neighbors = {}

    for term in tokens:
        if term in vocab_index:
            idx = vocab_index[term]
            row = assoc_matrix.getrow(idx)
            row_indices = row.indices
            row_data = row.data

            # create list of (term, score) excluding self
            term_scores = [(vocab[i], score) for i, score in zip(row_indices, row_data) if i != idx and score > 0]

            # sort descending by score and pick topk
            term_scores = sorted(term_scores, key=lambda x: x[1], reverse=True)[:topk]
            neighbors[term] = term_scores

    return neighbors

# Run for all queries
for q in queries_assignment3:
    neigh = get_neighborhood_terms_sparse(q, vocab_index, vocab, normalized_assoc_sparse, topk=5)
    print(f"\nQuery: {q}")
    for t, terms in neigh.items():
        print(f"  {t} -> {[w for w, s in terms]}")


Query: gas pressure
  gas -> ['equilibrium', 'air', 'injection', 'ideal', 'real']
  pressure -> ['number', 'mach', 'jet', 'flow', 'results']

Query: structural aeroelastic flight high speed aircraft
  structural -> ['loads', 'fatigue', 'aircraft', 'structure', 'random']
  aeroelastic -> ['thermo', 'piston', 'responses', 'stations', 'entirely']
  flight -> ['altitude', 'high', 'test', 'speed', 'range']
  high -> ['speed', 'may', 'numbers', 'speeds', 'effects']
  speed -> ['high', 'low', 'speeds', 'characteristics', 'effect']
  aircraft -> ['vtol', 'structural', 'ground', 'structure', 'slipstream']

Query: heat conduction composite slabs
  heat -> ['transfer', 'temperature', 'laminar', 'layer', 'boundary']
  conduction -> ['trail', 'controlled', 'radiation', 'solid', 'variational']
  composite -> ['slab', 'medium', 'periodic', 'refractory', 'slabs']
  slabs -> ['refractory', 'shielded', 'melting', 'composite', 'input']

Query: boundary layer control
  boundary -> ['layer', 'laminar', 'f